[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sharukh01621/first-line-to-first-paper/blob/main/colab_drive/part_05_classical_ml/ch25_unsupervised_clustering_pca/01_walkthrough.ipynb)

> **From First Line to First Paper** — *Mastering Python, Statistical Data Science, and Applied Deep Learning*  
> **Author:** Mezanur Rahman Sharukh | **Mode:** `INTERACTIVE STUDENT NOTEBOOK`


# Chapter 25 Walkthrough: Unsupervised Learning — Clustering & PCA

**Book Reference:** Chapter 25 (Part V — Classical Machine Learning), Sections 25.1 – 25.4  
Run each cell top-to-bottom (`Shift + Enter`) alongside the textbook dissections. Every workflow uses the bundled `clinic_triage` patient dataset from rural and urban Bangladesh Upazila Health Complexes.

In [ ]:
# @title Step 0: Click Play (▶) to Load Packages & Clinic Triage Data
!pip install -q "git+https://github.com/sharukh01621/first-line-to-first-paper.git#subdirectory=sharukh_tutor"
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score
import sharukh_tutor as tutor

clinic = tutor.load_data("clinic_triage", as_pandas=True)
feat_cols = [
    "age_years", "weight_kg", "height_cm",
    "temp_c", "systolic_bp", "hemoglobin_g_dl", "fever_days",
]
X_raw = clinic[feat_cols].to_numpy(dtype=float)
print("Loaded clinic_triage:", clinic.shape)
print("Feature matrix shape:", X_raw.shape)


## Section 25.3.1 — Why Feature Scaling Dominates Euclidean Distance

In [ ]:
# Inspect raw feature variances across incompatible clinical units
raw_vars = pd.Series(X_raw.var(axis=0), index=feat_cols).round(2)
print("Raw feature variances:\n" + raw_vars.to_string())
print(f"Variance ratio (age / temp): "
      f"{raw_vars['age_years'] / raw_vars['temp_c']:.1f}x")

X_scaled = StandardScaler().fit_transform(X_raw)
km_raw = KMeans(n_clusters=3, n_init=10, random_state=42)
km_scl = KMeans(n_clusters=3, n_init=10, random_state=42)
lbl_raw = km_raw.fit_predict(X_raw)
lbl_scl = km_scl.fit_predict(X_scaled)

sil_raw_in_scaled = silhouette_score(X_scaled, lbl_raw)
sil_scl_in_scaled = silhouette_score(X_scaled, lbl_scl)
print(f"Unscaled K-Means silhouette (scaled space): "
      f"{sil_raw_in_scaled:.4f}")
print(f"Scaled   K-Means silhouette (scaled space): "
      f"{sil_scl_in_scaled:.4f}")


## Section 25.3.2 — K-Means From Scratch in Pure NumPy

In [ ]:
def kmeans_scratch_demo(X, k, max_iter=50, seed=42):
    X_arr = np.asarray(X, dtype=float)
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(X_arr), size=k, replace=False)
    centroids = X_arr[idx].copy()
    for _ in range(max_iter):
        dists = ((X_arr[:, None, :] - centroids[None, :, :]) ** 2)
        labels = np.argmin(dists.sum(axis=2), axis=1).astype(int)
        new_c = centroids.copy()
        for cid in range(k):
            mask = labels == cid
            if np.any(mask):
                new_c[cid] = X_arr[mask].mean(axis=0)
        if np.allclose(new_c, centroids):
            centroids = new_c
            break
        centroids = new_c
    dists = ((X_arr[:, None, :] - centroids[None, :, :]) ** 2)
    labels = np.argmin(dists.sum(axis=2), axis=1).astype(int)
    inertia = float(np.sum((X_arr - centroids[labels]) ** 2))
    return labels, centroids, inertia

sc_labels, sc_centers, sc_inertia = kmeans_scratch_demo(
    X_scaled, k=3, max_iter=50, seed=42
)
print(f"Scratch K-Means inertia: {sc_inertia:.4f}")
print(f"Sklearn K-Means inertia: {km_scl.inertia_:.4f}")
print("Scratch cluster sizes:",
      {int(c): int(n) for c, n in zip(*np.unique(
          sc_labels, return_counts=True))})


## Section 25.3.3 — Choosing k: Elbow Method & Silhouette Score

In [ ]:
sweep_rows = []
for k in range(2, 7):
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    lbl = km.fit_predict(X_scaled)
    sweep_rows.append({
        "k": k,
        "inertia": round(float(km.inertia_), 4),
        "silhouette": round(float(silhouette_score(X_scaled, lbl)), 4),
    })
k_table = pd.DataFrame(sweep_rows)
print(k_table.to_string(index=False))
best_row = k_table.loc[k_table["silhouette"].idxmax()]
print(f"Peak silhouette at k={int(best_row['k'])} "
      f"({best_row['silhouette']:.4f})")


## Section 25.3.4 — Hierarchical Agglomerative Clustering (Ward Linkage)

In [ ]:
ward = AgglomerativeClustering(n_clusters=3, linkage="ward")
ward_labels = ward.fit_predict(X_scaled)
ward_sil = silhouette_score(X_scaled, ward_labels)
ward_sizes = {
    int(c): int(n)
    for c, n in zip(*np.unique(ward_labels, return_counts=True))
}
print(f"Ward (k=3) silhouette: {ward_sil:.4f}")
print("Ward (k=3) cluster sizes:", ward_sizes)


## Section 25.3.5 — DBSCAN Density Clustering & Anomaly Flagging

In [ ]:
db = DBSCAN(eps=1.8, min_samples=5)
db_labels = db.fit_predict(X_scaled)
n_clusters = int(len(set(db_labels.tolist()) - {-1}))
n_noise = int(np.sum(db_labels == -1))
print(f"DBSCAN (eps=1.8, min_samples=5): "
      f"{n_clusters} clusters, {n_noise} noise outliers")
outlier_preview = clinic.loc[
    db_labels == -1, ["patient_id", "age_years", "temp_c", "systolic_bp"]
].head(3)
print("Sample flagged clinical outliers:\n",
      outlier_preview.to_string(index=False))


## Section 25.3.6 — PCA From Scratch via NumPy SVD vs. Scikit-Learn

In [ ]:
# 1. PCA from scratch via centered SVD
X_c = X_scaled - X_scaled.mean(axis=0)
U, S, Vt = np.linalg.svd(X_c, full_matrices=False)
Z_svd = X_c @ Vt[:2].T
evr_svd = ((S ** 2) / np.sum(S ** 2))[:2]

# 2. Compare directly against sklearn.decomposition.PCA
pca_full = PCA().fit(X_scaled)
cum_var = np.cumsum(pca_full.explained_variance_ratio_)
print("SVD EVR (top 2):    ", np.round(evr_svd, 4))
print("Sklearn EVR (top 2):",
      np.round(pca_full.explained_variance_ratio_[:2], 4))
print("Cumulative variance:", np.round(cum_var, 4))
k_85 = int(np.argmax(cum_var >= 0.85) + 1)
print(f"PCs required for >=85% variance: {k_85} "
      f"(retains {cum_var[k_85 - 1]:.4f})")

# 3. Interpret clinical loadings on PC1 & PC2
loadings = pd.DataFrame(
    pca_full.components_[:2].T,
    index=feat_cols,
    columns=["PC1", "PC2"],
).round(4)
print("Top-2 PC Clinical Loadings:\n", loadings)


## Section 25.3.7 — Nonlinear Projections (t-SNE / UMAP) Caveats

In [ ]:
# t-SNE preserves local neighborhoods, NOT global distances or variance
for perp in [10, 30]:
    emb = TSNE(
        n_components=2, perplexity=perp, init="pca",
        learning_rate="auto", random_state=42,
    ).fit_transform(X_scaled)
    spread = [round(float(v), 2) for v in emb.std(axis=0)]
    print(f"t-SNE (perplexity={perp:2d}) embedding shape={emb.shape}, "
          f"axis SDs={spread}")


## Section 25.3.8 — Break It On Purpose! (Unsupervised Error Anatomy)

In [ ]:
# Intentional Bug #1: ValueError — silhouette_score with only 1 cluster
all_one_cluster = np.zeros(len(X_scaled), dtype=int)
silhouette_score(X_scaled, all_one_cluster)


In [ ]:
# Intentional Bug #2: ValueError — n_components > min(n_samples, p)
PCA(n_components=12).fit(X_scaled)


In [ ]:
# Intentional Bug #3: ValueError — negative DBSCAN eps radius
DBSCAN(eps=-0.5, min_samples=5).fit(X_scaled)


## Section 25.4 — Stage 4 Blueprint: End-to-End Unsupervised Clinical Phenotyping Pipeline

In [ ]:
def run_clinical_phenotyping_pipeline(
    raw_df, feat_cols, k=3, var_target=0.85, eps=1.8, seed=42
):
    """Scale vitals, audit PCA compression, and compare clusterers."""
    X = raw_df[feat_cols].to_numpy(dtype=float)
    X_sc = StandardScaler().fit_transform(X)

    # 1. PCA compression & reconstruction MSE audit
    full_pca = PCA().fit(X_sc)
    cum = np.cumsum(full_pca.explained_variance_ratio_)
    n_pcs = int(np.argmax(cum >= var_target) + 1)
    pca_k = PCA(n_components=n_pcs)
    X_hat = pca_k.inverse_transform(pca_k.fit_transform(X_sc))
    recon_mse = round(float(np.mean((X_sc - X_hat) ** 2)), 5)

    # 2. Compare K-Means vs. Ward Agglomerative at k clusters
    km_lbl = KMeans(
        n_clusters=k, n_init=10, random_state=seed
    ).fit_predict(X_sc)
    ward_lbl = AgglomerativeClustering(
        n_clusters=k, linkage="ward"
    ).fit_predict(X_sc)

    # 3. Flag density anomalies via DBSCAN
    db_lbl = DBSCAN(eps=eps, min_samples=5).fit_predict(X_sc)
    return {
        "n_pcs": n_pcs,
        "cum_variance": round(float(cum[n_pcs - 1]), 4),
        "recon_mse": recon_mse,
        "kmeans_sil": round(float(silhouette_score(X_sc, km_lbl)), 4),
        "ward_sil": round(float(silhouette_score(X_sc, ward_lbl)), 4),
        "dbscan_noise": int(np.sum(db_lbl == -1)),
    }

audit = run_clinical_phenotyping_pipeline(clinic, feat_cols)
for key, val in audit.items():
    print(f"{key:14s}: {val}")
